# Flood Mapping Pipeline: Auckland Anniversary Floods (27 Jan 2023)

GEOG761 Group 05 — Flood Mapping & Response

**Event-based version of the pipeline.** The earlier notebook (`flood_data_pipeline.ipynb`) pulled a rolling 24-day Sentinel-1 median for all of NZ. A median over weeks removes a short-lived flood, and there was no pre-event image to compare against. This notebook fixes that and follows the methods taught in the course.

| Step | What we do | Course link |
|---|---|---|
| 1 | Find Sentinel-1 scenes before/after the event (same orbit) | Lecture 6 (S1 GRD, IW, VV/VH) |
| 2 | Speckle filter + VV/VH feature | Lecture 6, Exercise 2 and 3 |
| 3 | Random Forest water classifier, labels from ESA WorldCover, 70/20/10 split | Lecture 6 Ex.1, Lab 2 |
| 4 | Otsu threshold as a simple baseline | Pitch slide 7 (Otsu vs RF vs CNN) |
| 5 | Flood = water after event AND NOT water before AND NOT permanent water AND flat terrain | Lecture 6 (radar shadow looks like water) |
| 6 | Accuracy assessment (confusion matrix, precision, recall, F1) + optional independent reference | Lab 2, Lab 5 |
| 7 | Depth: "bathtub" from the DEM at the flood boundary | Pitch slide 6 (not covered in lectures) |
| 8 | Exposure: population, critical and transport infrastructure | Pitch slides 5-6 |

Run top to bottom. **Check the printed Sentinel-1 table in section 3 before trusting any result.**

## 1. Setup

In [ ]:
!pip install -q geemap geedim osmnx geopandas rasterio

In [ ]:
import ee

# One-time interactive login per Colab session.
ee.Authenticate()

# GEE-registered Cloud project. If you get a permission error, replace this with your own project (Lab notebooks do the same).
EE_PROJECT = "geog761-labs"
ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready, project:", EE_PROJECT)

## 2. Configuration

Everything you might want to change is in this one cell.

In [ ]:
from datetime import date, timedelta
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geemap

# --- Study area: Auckland region (west, south, east, north) -------------------
AOI_BBOX = (174.55, -37.05, 175.00, -36.65)
AOI = ee.Geometry.Rectangle(list(AOI_BBOX))

# --- Event ------------------------------------------------------------------------
EVENT_NAME = "Auckland Anniversary floods"
EVENT_DATE = date(2023, 1, 27)   # heavy rain from the afternoon/evening of 27 Jan (NZDT)
BASELINE_DAYS = 30               # how far back to look for pre-event scenes
POST_DAYS = 14                   # how far forward to look (Sentinel-1 revisit over NZ can be ~6-12 days)

BASE_START = str(EVENT_DATE - timedelta(days=BASELINE_DAYS))
BASE_END = str(EVENT_DATE)                       # baseline stops at the start of 27 Jan (UTC)
# Post-event scenes must be AFTER the rain peak. Sentinel-1 times are UTC; NZDT = UTC+13.
# ASSUMPTION: rain peaked in the evening of 27 Jan NZDT, so we accept scenes from 27 Jan 00:00 UTC onward (the table shows NZ time so you can judge).
# Check the table in section 3 and edit this if needed.
POST_START = f"{EVENT_DATE}T00:00:00"
POST_END = str(EVENT_DATE + timedelta(days=POST_DAYS))

# Leave as None to pick automatically from the first post-event scene (recommended).
ORBIT_PASS = None    # "ASCENDING" or "DESCENDING"
REL_ORBIT = None     # relative orbit number

# --- Processing parameters -------------------------------------------------------
SCALE_M = 30            # analysis resolution (m). Lower to 10 for a smaller AOI.
SPECKLE_RADIUS_M = 50   # boxcar speckle filter radius (Lecture 6 used 50 m)
SAMPLES_PER_CLASS = 2000
RF_TREES = 100          # Lab 2 used 100 trees
SEED = 42
SLOPE_MAX_DEG = 5       # steeper than this: radar shadow / not floodable
ELEV_MAX_M = 100        # higher than this: not floodable
MIN_PATCH_PIXELS = 8    # drop flood clusters smaller than this (noise)
WSE_RADIUS_M = 500      # neighbourhood for water-surface elevation (depth step)

OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("AOI:", AOI_BBOX)
print("Baseline:", BASE_START, "to", BASE_END, "| Post-event:", POST_START, "to", POST_END)

## 3. Find Sentinel-1 scenes

Same instrument mode as Lecture 6 (IW, VV + VH, GRD). A flood comparison is only valid between scenes from the **same orbit direction and relative orbit**, because look angle changes how the same surface appears (Lecture 6, "Look Angle"). The code picks the orbit from the first post-event scene.

In [ ]:
def s1_collection(aoi, start, end, orbit_pass=None, rel_orbit=None):
    col = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    )
    if orbit_pass:
        col = col.filter(ee.Filter.eq("orbitProperties_pass", orbit_pass))
    if rel_orbit is not None:
        col = col.filter(ee.Filter.eq("relativeOrbitNumber_start", rel_orbit))
    return col


def list_acquisitions(col):
    df = pd.DataFrame({
        "t": col.aggregate_array("system:time_start").getInfo(),
        "pass": col.aggregate_array("orbitProperties_pass").getInfo(),
        "rel_orbit": col.aggregate_array("relativeOrbitNumber_start").getInfo(),
        "id": col.aggregate_array("system:index").getInfo(),
    })
    if df.empty:
        return df
    df["utc"] = pd.to_datetime(df["t"], unit="ms", utc=True)
    df["nz_time"] = df["utc"].dt.tz_convert("Pacific/Auckland")
    return df.sort_values("utc")[["id", "pass", "rel_orbit", "utc", "nz_time"]].reset_index(drop=True)


print("Diagnostic: ALL Sentinel-1 IW scenes over the AOI, 20 Jan - 20 Feb 2023:")
display(list_acquisitions(s1_collection(AOI, "2023-01-20", "2023-02-20")))
print("Baseline window:")
base_all_df = list_acquisitions(s1_collection(AOI, BASE_START, BASE_END))
display(base_all_df)
print("Post-event window:")
post_all_df = list_acquisitions(s1_collection(AOI, POST_START, POST_END))
display(post_all_df)

if post_all_df.empty:
    raise RuntimeError("No post-event Sentinel-1 scene found. Check the diagnostic table above and set POST_START / POST_END to a date range that contains a scene.")

# Pick the first post-event scene whose orbit also has baseline scenes (same look angle).
if ORBIT_PASS is None or REL_ORBIT is None:
    for _, r in post_all_df.iterrows():
        ok = ((base_all_df["pass"] == r["pass"]) & (base_all_df["rel_orbit"] == r["rel_orbit"])).any() if not base_all_df.empty else False
        if ok:
            ORBIT_PASS, REL_ORBIT = r["pass"], int(r["rel_orbit"])
            break
    else:
        raise RuntimeError("No post-event orbit has baseline scenes. Increase BASELINE_DAYS.")
print("Using orbit:", ORBIT_PASS, "| relative orbit:", REL_ORBIT)

base_col = s1_collection(AOI, BASE_START, BASE_END, ORBIT_PASS, REL_ORBIT)
post_col = s1_collection(AOI, POST_START, POST_END, ORBIT_PASS, REL_ORBIT)
n_base, n_post = base_col.size().getInfo(), post_col.size().getInfo()
print(f"Scenes used -> baseline: {n_base}, post-event: {n_post}")
if n_base == 0:
    raise RuntimeError("No baseline scene on the same orbit. Increase BASELINE_DAYS or set ORBIT_PASS/REL_ORBIT manually.")

## 4. Speckle filter and features

Lecture 6, Exercise 3: raw SAR is speckled, which makes land pixels look dark and hurts classification. We apply a simple boxcar filter (`focal_mean`). Exercise 2 suggests VV - VH (a ratio in dB) as an extra feature.

In [ ]:
def despeckle(img, radius_m=SPECKLE_RADIUS_M):
    return img.focal_mean(radius=radius_m, units="meters")


def add_features(img):
    ratio = img.select("VV").subtract(img.select("VH")).rename("VV_VH")
    return img.addBands(ratio)


def make_composite(col, aoi):
    # Values are already in dB in COPERNICUS/S1_GRD; mean of the scenes in the window, as in Lecture 6.
    return add_features(despeckle(col.select(["VV", "VH"]).mean())).clip(aoi)


base_img = make_composite(base_col, AOI)
post_img = make_composite(post_col, AOI)
FEATURES = ["VV", "VH", "VV_VH"]

Map = geemap.Map(center=[-36.85, 174.78], zoom=10, basemap="HYBRID")
Map.addLayer(base_img.select("VV"), {"min": -25, "max": 0}, "VV baseline (pre-event)")
Map.addLayer(post_img.select("VV"), {"min": -25, "max": 0}, "VV post-event")
Map.addLayer(AOI, {"color": "red"}, "AOI", False)
Map

## 5. Random Forest water classifier

Same recipe as Lecture 6 Exercise 1 and Lab 2:

- labels from ESA WorldCover (water = class 80), the same "automatic training data" trick as the lab;
- 70/20/10 train/validation/test split with `randomColumn`;
- `ee.Classifier.smileRandomForest`.

We train on the **baseline (pre-event) image**, so the labels describe permanent conditions and are not contaminated by flood water. Then we apply the classifier to both the baseline and the post-event image.

Known limitation (same as in Lab 2): WorldCover is itself satellite-derived, and most of the "water" in this AOI is harbour and sea, not inland flood water. Accuracy here measures water-vs-land separation, not flood accuracy. See section 8.

In [ ]:
worldcover = ee.Image("ESA/WorldCover/v200/2021").select("Map").clip(AOI)

# 1 = water, 0 = land. Wetland (90) and mangrove (95) are ambiguous for SAR water, so they are left out of training.
label = (
    worldcover.eq(80).rename("label")
    .updateMask(worldcover.neq(90).And(worldcover.neq(95)))
)

train_stack = base_img.select(FEATURES).addBands(label)
samples = train_stack.stratifiedSample(
    numPoints=SAMPLES_PER_CLASS,
    classBand="label",
    region=AOI,
    scale=SCALE_M,
    seed=SEED,
    classValues=[0, 1],
    classPoints=[SAMPLES_PER_CLASS, SAMPLES_PER_CLASS],
    geometries=True,
)

# 70 / 20 / 10 split, as in Lab 2
samples = samples.randomColumn("random", SEED)
train = samples.filter(ee.Filter.lt("random", 0.7))
valid = samples.filter(ee.Filter.And(ee.Filter.gte("random", 0.7), ee.Filter.lt("random", 0.9)))
test = samples.filter(ee.Filter.gte("random", 0.9))
print("train / valid / test:", train.size().getInfo(), valid.size().getInfo(), test.size().getInfo())

rf = ee.Classifier.smileRandomForest(numberOfTrees=RF_TREES, seed=SEED).train(
    features=train, classProperty="label", inputProperties=FEATURES
)
print("Feature importance:", rf.explain().get("importance").getInfo())

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report


def report_binary(y_true, y_pred, name, class_names=("Land", "Water")):
    y_true = np.array(y_true).astype(int)
    y_pred = np.array(y_pred).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"=== {name} ===")
    print(pd.DataFrame(cm, index=[f"Actual {c}" for c in class_names], columns=[f"Pred {c}" for c in class_names]))
    print(classification_report(y_true, y_pred, labels=[0, 1], target_names=list(class_names), digits=3))
    tp, fp, fn = cm[1, 1], cm[0, 1], cm[1, 0]
    iou = tp / (tp + fp + fn) if (tp + fp + fn) else float("nan")
    print(f"IoU (positive class): {iou:.3f}\n")
    return {"accuracy": float((y_true == y_pred).mean()), "iou": float(iou)}


def evaluate_fc(fc, classifier, name):
    pred = fc.classify(classifier, "predicted")
    y_true = pred.aggregate_array("label").getInfo()
    y_pred = pred.aggregate_array("predicted").getInfo()
    return report_binary(y_true, y_pred, name)


rf_valid_metrics = evaluate_fc(valid, rf, "Random Forest, validation split (20%)")
rf_test_metrics = evaluate_fc(test, rf, "Random Forest, test split (10%)")

In [ ]:
water_base = base_img.select(FEATURES).classify(rf).rename("water")
water_post = post_img.select(FEATURES).classify(rf).rename("water")

## 6. Otsu threshold baseline

The pitch (slide 7) lists Otsu thresholding as the simplest method, used in operational systems such as Copernicus GFM. We compute it here so Random Forest can be compared against it, the same way the example reports compared more than one model.

In [ ]:
def otsu(histogram):
    counts = ee.Array(ee.Dictionary(histogram).get("histogram"))
    means = ee.Array(ee.Dictionary(histogram).get("bucketMeans"))
    size = means.length().get([0])
    total = counts.reduce(ee.Reducer.sum(), [0]).get([0])
    total_sum = means.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0])
    mean = total_sum.divide(total)
    indices = ee.List.sequence(1, size)

    def between_class_variance(i):
        a_counts = counts.slice(0, 0, i)
        a_count = a_counts.reduce(ee.Reducer.sum(), [0]).get([0])
        a_means = means.slice(0, 0, i)
        a_mean = a_means.multiply(a_counts).reduce(ee.Reducer.sum(), [0]).get([0]).divide(a_count)
        b_count = total.subtract(a_count)
        b_mean = total_sum.subtract(a_count.multiply(a_mean)).divide(b_count)
        return (
            a_count.multiply(a_mean.subtract(mean).pow(2))
            .add(b_count.multiply(b_mean.subtract(mean).pow(2)))
        )

    bss = indices.map(between_class_variance)
    return means.sort(bss).get([-1])


def otsu_threshold(image, band, region, scale):
    hist = image.select(band).reduceRegion(
        reducer=ee.Reducer.histogram(255, 0.1),
        geometry=region, scale=scale, maxPixels=1e10, bestEffort=True,
    )
    return ee.Number(otsu(hist.get(band)))


otsu_vv = otsu_threshold(post_img, "VV", AOI, SCALE_M).getInfo()
print(f"Otsu threshold on post-event VV: {otsu_vv:.2f} dB")

water_post_otsu = post_img.select("VV").lt(otsu_vv).rename("water")
water_base_otsu = base_img.select("VV").lt(otsu_vv).rename("water")

## 7. Flood extent

A pixel is flooded if it is **water after the event, not water before it, not permanent water, and on flat low ground**. The terrain mask matters because radar shadow on steep slopes is also dark (Lecture 6, "Shadowing"). Elevation is Copernicus GLO-30 (the same DEM used for depth below).

In [ ]:
dem_ic = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM")
dem_proj = dem_ic.first().projection()
elevation = dem_ic.mosaic().setDefaultProjection(dem_proj).rename("elevation_m").clip(AOI)
slope = ee.Terrain.slope(elevation)

# Permanent / seasonal water: WorldCover class 80 plus JRC Global Surface Water occurrence > 10 %
jrc_occurrence = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").unmask(0)
permanent_water = worldcover.eq(80).Or(jrc_occurrence.gt(10))
terrain_ok = slope.lt(SLOPE_MAX_DEG).And(elevation.lt(ELEV_MAX_M))


def flood_from_water(w_post, w_base):
    f = (
        w_post.eq(1).And(w_base.eq(0))
        .And(permanent_water.Not())
        .And(terrain_ok)
    )
    size = f.selfMask().connectedPixelCount(100, True)
    keep = size.gte(MIN_PATCH_PIXELS).unmask(0)
    return f.And(keep).rename("flood").toByte()


flood_rf = flood_from_water(water_post, water_base)
flood_otsu = flood_from_water(water_post_otsu, water_base_otsu)


def area_km2(mask):
    a = mask.multiply(ee.Image.pixelArea()).reduceRegion(
        ee.Reducer.sum(), AOI, SCALE_M, maxPixels=1e10, bestEffort=True
    ).get("flood")
    return ee.Number(a).divide(1e6).getInfo()


area_rf, area_otsu = area_km2(flood_rf), area_km2(flood_otsu)
both = area_km2(flood_rf.And(flood_otsu).rename("flood"))
either = area_km2(flood_rf.Or(flood_otsu).rename("flood"))
print(f"Flooded area, Random Forest: {area_rf:.1f} km2")
print(f"Flooded area, Otsu:          {area_otsu:.1f} km2")
print(f"Agreement between methods (IoU): {both / either:.2f}" if either else "No flood detected")

Map = geemap.Map(center=[-36.85, 174.78], zoom=10, basemap="HYBRID")
Map.addLayer(post_img.select("VV"), {"min": -25, "max": 0}, "VV post-event", False)
Map.addLayer(flood_otsu.selfMask(), {"palette": ["red"]}, "Flood (Otsu)")
Map.addLayer(flood_rf.selfMask(), {"palette": ["cyan"]}, "Flood (Random Forest)")
Map

## 8. Accuracy assessment

Two levels:

1. **Water classification** (section 5): confusion matrix, precision, recall and F1 on the held-out validation and test splits. This is what Lab 2 and Lecture 6 do. The labels are not independent of the satellite data, so treat this as a sanity check.
2. **Flood extent against an independent reference.** The report requires an accuracy assessment, so we need reference flood data that did not come from our own classifier, for example an official flood extent or a hand-digitised extent from post-event imagery. Fill in the cell below when you have it. Until then, do not claim a flood-extent accuracy in the presentation or report.

The cell after it shows Sentinel-2 imagery for visual checking. Cloud cover during the event will limit this.

In [ ]:
# Reference data. Provide GeoJSON files (upload to Colab) with polygons that are
# (a) known to be flooded and (b) known to be dry. Leave as None to skip.
REF_FLOODED_GEOJSON = None   # e.g. "/content/ref_flooded.geojson"
REF_DRY_GEOJSON = None       # e.g. "/content/ref_dry.geojson"


def assess_against_reference(flood_img, flooded_geom, dry_geom, n=500, name="Flood"):
    pts_f = ee.FeatureCollection.randomPoints(flooded_geom, n, SEED).map(lambda f: f.set("ref", 1))
    pts_d = ee.FeatureCollection.randomPoints(dry_geom, n, SEED + 1).map(lambda f: f.set("ref", 0))
    sampled = flood_img.sampleRegions(collection=pts_f.merge(pts_d), properties=["ref"], scale=SCALE_M)
    y_true = sampled.aggregate_array("ref").getInfo()
    y_pred = sampled.aggregate_array("flood").getInfo()
    return report_binary(y_true, y_pred, name, class_names=("Dry", "Flooded"))


ref_metrics = {}
if REF_FLOODED_GEOJSON and REF_DRY_GEOJSON:
    flooded_geom = geemap.geojson_to_ee(REF_FLOODED_GEOJSON).geometry()
    dry_geom = geemap.geojson_to_ee(REF_DRY_GEOJSON).geometry()
    ref_metrics["rf"] = assess_against_reference(flood_rf, flooded_geom, dry_geom, name="Flood extent, Random Forest")
    ref_metrics["otsu"] = assess_against_reference(flood_otsu, flooded_geom, dry_geom, name="Flood extent, Otsu")
else:
    print("No independent reference supplied: flood-extent accuracy NOT assessed.")

In [ ]:
# Optional visual check with Sentinel-2 (true colour) around the event
def s2_true_colour(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(AOI).filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
        .median().clip(AOI)
    )

vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
Map = geemap.Map(center=[-36.85, 174.78], zoom=11, basemap="HYBRID")
Map.addLayer(s2_true_colour(BASE_START, BASE_END), vis, "S2 before")
Map.addLayer(s2_true_colour(str(EVENT_DATE), POST_END), vis, "S2 after (cloud-limited)")
Map.addLayer(flood_rf.selfMask(), {"palette": ["cyan"]}, "Flood (Random Forest)")
Map

## 9. Flood depth: bathtub model from the flood boundary

Pitch slide 6: depth = water-surface elevation − ground elevation, where the water surface is taken from the DEM at the flood boundary. The method here:

1. take the DEM at the dry pixels just outside the flooded area (the boundary);
2. the local water-surface elevation is the **median boundary elevation within `WSE_RADIUS_M`**;
3. depth = water-surface elevation − ground elevation, floored at 0, only inside the flood.

This replaces the earlier "highest ground in each patch" rule, which over-estimates depth, and avoids the connected-components size limit. It is a simple bathtub model, not a hydraulic simulation. Assumptions to state in the report: flat water surface within the radius, GLO-30 is a surface model (buildings and trees included) at 30 m, and results are only as good as the flood extent.

In [ ]:
def bathtub_depth(flood, ground, radius_m=WSE_RADIUS_M, scale=SCALE_M):
    flood = flood.reproject(crs="EPSG:2193", scale=scale)
    ground = ground.reproject(crs="EPSG:2193", scale=scale)
    # dry pixels touching the flood = boundary
    boundary = flood.focal_max(radius=1, kernelType="square", units="pixels").And(flood.Not())
    boundary_elev = ground.updateMask(boundary)
    wse = boundary_elev.reduceNeighborhood(
        reducer=ee.Reducer.median(),
        kernel=ee.Kernel.circle(radius=radius_m, units="meters"),
    ).rename("wse_m")
    depth = wse.subtract(ground).max(0).updateMask(flood).rename("depth_m")
    return depth


depth_rf = bathtub_depth(flood_rf, elevation)

stats = depth_rf.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.percentile([50, 95]), sharedInputs=True),
    geometry=AOI, scale=SCALE_M, maxPixels=1e10, bestEffort=True,
).getInfo()
print("Depth statistics (m):", {k: round(v, 2) for k, v in stats.items() if v is not None})

Map = geemap.Map(center=[-36.85, 174.78], zoom=10, basemap="HYBRID")
Map.addLayer(depth_rf, {"min": 0, "max": 3, "palette": ["#deebf7", "#6baed6", "#08306b"]}, "Depth (m)")
Map

## 10. Population exposure

WorldPop 100 m (2020, latest year in the collection). Flooded fraction of each 100 m cell × cell population gives an approximate exposed population.

In [ ]:
pop_ic = (
    ee.ImageCollection("WorldPop/GP/100m/pop")
    .filter(ee.Filter.eq("country", "NZL"))
    .filter(ee.Filter.eq("year", 2020))
)
pop_proj = pop_ic.first().projection()
pop = pop_ic.mosaic().setDefaultProjection(pop_proj).clip(AOI)

flood_frac = flood_rf.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024).reproject(pop_proj)
exposed = pop.multiply(flood_frac)

def sum_image(img):
    d = img.reduceRegion(ee.Reducer.sum(), AOI, 100, crs=pop_proj, maxPixels=1e10, tileScale=16)  # tileScale: avoids 'User memory limit exceeded'
    return list(d.getInfo().values())[0]

total_pop = sum_image(pop)
exposed_pop = sum_image(exposed)
print(f"Population in AOI: {total_pop:,.0f}")
print(f"Population in flooded area (approx.): {exposed_pop:,.0f} ({100 * exposed_pop / total_pop:.1f} %)")

## 11. Export rasters

Exports are in NZTM (EPSG:2193) so pixel areas and lengths are in metres. `geemap.download_ee_image` tiles large requests automatically.

In [ ]:
EXPORTS = {
    "vv_baseline": base_img.select("VV"),
    "vv_post": post_img.select("VV"),
    "flood_rf": flood_rf,
    "flood_otsu": flood_otsu,
    "depth_rf": depth_rf,
}
paths = {}
for name, img in EXPORTS.items():
    paths[name] = str(OUT_DIR / f"{name}.tif")
    geemap.download_ee_image(img, filename=paths[name], region=AOI, scale=SCALE_M, crs="EPSG:2193")
    print("Saved:", paths[name])

## 12. Infrastructure exposure (OpenStreetMap)

Critical and transport infrastructure for the AOI from OpenStreetMap (LINZ / Auckland Transport would be the preferred sources in the pitch but need separate access). Roads are limited to the major network to keep the query manageable. Overpass is a shared public service and sometimes refuses connections; the code retries on mirrors.

In [ ]:
import time
import geopandas as gpd
import osmnx as ox
import rasterio
from rasterio.features import shapes
from shapely.geometry import box, shape as shapely_shape

try:
    from osmnx._errors import InsufficientResponseError
except ImportError:
    class InsufficientResponseError(Exception):
        pass

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.fr/api/interpreter",
]
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True
AOI_POLY = box(*AOI_BBOX)

CRITICAL_TAGS = {
    "hospitals_clinics": {"amenity": ["hospital", "clinic"]},
    "emergency_services": {"amenity": ["fire_station", "police"]},
    "schools": {"amenity": ["school"]},
    "power_substations": {"power": ["substation", "plant"]},
}
TRANSPORT_TAGS = {
    "major_roads": {"highway": ["motorway", "trunk", "primary", "secondary", "tertiary"]},
    "railways": {"railway": ["rail"]},
}
LENGTH_LAYERS = {"major_roads", "railways"}   # measured in km; everything else is counted


def fetch_osm(tags):
    last = None
    for attempt, endpoint in enumerate(OVERPASS_ENDPOINTS):
        ox.settings.overpass_endpoint = endpoint
        try:
            return ox.features_from_polygon(AOI_POLY, tags)
        except InsufficientResponseError:
            return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
        except Exception as exc:
            last = exc
            time.sleep(2 * (attempt + 1))
    print("  [warn] all Overpass mirrors failed:", repr(last))
    return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")


layers = {}
for name, tags in {**CRITICAL_TAGS, **TRANSPORT_TAGS}.items():
    print("fetching", name)
    layers[name] = fetch_osm(tags)
    print(f"  {len(layers[name])} features")

In [ ]:
# Vectorise the flood raster
with rasterio.open(paths["flood_rf"]) as src:
    arr = src.read(1)
    flood_crs = src.crs
    flood_polys = [
        shapely_shape(geom)
        for geom, val in shapes(arr, mask=(arr == 1), transform=src.transform)
    ]
flood_gdf = gpd.GeoDataFrame(geometry=flood_polys, crs=flood_crs)
flood_union = flood_gdf.geometry.union_all() if hasattr(flood_gdf.geometry, "union_all") else flood_gdf.unary_union
print("Flood polygons:", len(flood_gdf))


def exposure_table(layers):
    rows = []
    for name, gdf in layers.items():
        if gdf is None or gdf.empty:
            rows.append({"layer": name, "total": 0, "affected": 0, "unit": "km" if name in LENGTH_LAYERS else "features"})
            continue
        gdf = gdf.to_crs(flood_crs)
        if name in LENGTH_LAYERS:
            gdf = gdf[gdf.geom_type.isin(["LineString", "MultiLineString"])]
            total_km = gdf.length.sum() / 1000
            hit = gdf[gdf.intersects(flood_union)]
            hit_km = hit.geometry.intersection(flood_union).length.sum() / 1000
            rows.append({"layer": name, "total": round(total_km, 1), "affected": round(hit_km, 1), "unit": "km"})
        else:
            hit = gdf[gdf.intersects(flood_union)]
            rows.append({"layer": name, "total": len(gdf), "affected": len(hit), "unit": "features"})
    return pd.DataFrame(rows)


exposure = exposure_table(layers)
exposure.to_csv(OUT_DIR / "infrastructure_exposure.csv", index=False)
exposure

## 13. Figures and summary numbers

Figures are saved at 300 dpi in `/content/outputs`. Check labels and captions before using them in the presentation or report.

In [ ]:
import matplotlib.pyplot as plt

def read_raster(path, masked_zero=False):
    with rasterio.open(path) as src:
        a = src.read(1, masked=True).astype("float32")
        b = src.bounds
    if masked_zero:
        a = np.ma.masked_where(a == 0, a)
    return a, (b.left, b.right, b.bottom, b.top)

vv0, ext = read_raster(paths["vv_baseline"])
vv1, _ = read_raster(paths["vv_post"])
fl_rf, _ = read_raster(paths["flood_rf"], masked_zero=True)
fl_ot, _ = read_raster(paths["flood_otsu"], masked_zero=True)
dep, _ = read_raster(paths["depth_rf"])

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
panels = [
    (axes[0, 0], vv0, "Sentinel-1 VV, before (dB)", dict(cmap="gray", vmin=-25, vmax=0)),
    (axes[0, 1], vv1, "Sentinel-1 VV, after (dB)", dict(cmap="gray", vmin=-25, vmax=0)),
]
for ax, data, title, kw in panels:
    im = ax.imshow(data, extent=ext, **kw)
    ax.set_title(title, fontsize=13)
    fig.colorbar(im, ax=ax, shrink=0.7)

ax = axes[0, 2]
ax.imshow(vv1, extent=ext, cmap="gray", vmin=-25, vmax=0)
ax.imshow(fl_ot, extent=ext, cmap="autumn", alpha=0.9)
ax.set_title("Flood extent, Otsu", fontsize=13)

ax = axes[1, 0]
ax.imshow(vv1, extent=ext, cmap="gray", vmin=-25, vmax=0)
ax.imshow(fl_rf, extent=ext, cmap="winter", alpha=0.9)
ax.set_title("Flood extent, Random Forest", fontsize=13)

ax = axes[1, 1]
ax.imshow(vv1, extent=ext, cmap="gray", vmin=-25, vmax=0)
im = ax.imshow(dep, extent=ext, cmap="Blues", vmin=0, vmax=3)
ax.set_title("Estimated flood depth (m)", fontsize=13)
fig.colorbar(im, ax=ax, shrink=0.7)

ax = axes[1, 2]
ex = exposure.set_index("layer")
ax.barh(ex.index, ex["total"], alpha=0.35, label="Total in AOI")
ax.barh(ex.index, ex["affected"], label="Intersecting flood")
ax.set_title("Infrastructure exposure (counts; km for roads/rail)", fontsize=13)
ax.legend()

for a in axes.ravel()[:5]:
    a.set_xlabel("Easting (m, NZTM)")
    a.set_ylabel("Northing (m, NZTM)")
fig.suptitle(f"{EVENT_NAME}: {EVENT_DATE}", fontsize=16)
fig.tight_layout()
fig_path = OUT_DIR / "figure_overview.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print("Saved:", fig_path)

In [ ]:
summary = {
    "event": EVENT_NAME,
    "event_date": str(EVENT_DATE),
    "orbit": {"pass": ORBIT_PASS, "relative_orbit": REL_ORBIT},
    "scenes_used": {"baseline": n_base, "post_event": n_post},
    "scale_m": SCALE_M,
    "otsu_vv_threshold_db": round(otsu_vv, 2),
    "rf_water_classification": {"validation": rf_valid_metrics, "test": rf_test_metrics},
    "flood_area_km2": {"random_forest": round(area_rf, 1), "otsu": round(area_otsu, 1)},
    "rf_otsu_agreement_iou": round(both / either, 3) if either else None,
    "independent_reference_metrics": ref_metrics or "not assessed",
    "population": {"total_aoi": round(total_pop), "exposed_approx": round(exposed_pop)},
    "depth_stats_m": {k: round(v, 2) for k, v in stats.items() if v is not None},
}
with open(OUT_DIR / "results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 14. Data sources (for the report table)

| Data | Source | Resolution | Use |
|---|---|---|---|
| Sentinel-1 GRD (IW, VV+VH) | ESA Copernicus via Google Earth Engine | 10 m (analysed at `SCALE_M`) | Flood detection, before and after the event |
| Land cover | ESA WorldCover v200 (2021) | 10 m | Water/land training labels, permanent-water mask |
| Surface water occurrence | JRC Global Surface Water v1.4 | 30 m | Permanent/seasonal water mask |
| Elevation | Copernicus GLO-30 DEM | 30 m | Terrain mask, depth (bathtub) |
| Population | WorldPop 100 m (2020) | 100 m | Exposed population |
| Infrastructure | OpenStreetMap (Overpass) | Vector | Critical and transport infrastructure exposure |
| Sentinel-2 SR (optional) | ESA Copernicus via GEE | 10 m | Visual check only |

## 15. Known limitations (write these into the discussion)

- No independent flood reference yet: flood-extent accuracy is not assessed until section 8 is filled in.
- RF labels come from WorldCover and training water is mostly harbour/sea; the classifier is a water detector, and the flood is the change between two dates.
- Sentinel-1 revisit (likely ~12 days in 2023): the post-event scene may not be at flood peak. Water may already have receded.
- SAR struggles with flooded vegetation and urban areas (double bounce raises backscatter), so flooding in built-up areas is probably under-detected.
- GLO-30 is a 30 m surface model; the bathtub depth is approximate.
- OSM coverage and completeness vary; WorldPop is a 2020 estimate.